In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict, KFold
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
## load internal (in-house) + public-augmented data, champion model, eval helpers
from copy import deepcopy



# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from python.harmonize_sol import harmonize_solubility

from get_library import get_df   # CDD Vault collection export
# from tdc.multi_pred import DTI

In [ ]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')


## 0. Imports

In [ ]:
## load training file:
# zinc = pd.read_csv('data/protacdb2.0_zinc_chembl_dataset.csv')
# zinc.head(1)

In [ ]:
## get harmonized library
lib = harmonize_solubility(out_csv='data/public_solubility/harmonized_sol.csv')
lib['compound'] = 'X' + lib.index.astype(str)

## compute H236 fingerprints of training set:
if SOL_OVERWRITE:
    MF_features_train = rdkit_tools.compute_H236_features(lib, v=True)
    MF_features_train.to_parquet('autoresearch/predict_solubility/training_MF_features.parquet')
    lib.to_parquet('autoresearch/predict_solubility/training.parquet')
else:
    MF_features_train = pd.read_parquet('autoresearch/predict_solubility/training_MF_features.parquet')

print(lib.shape, '|', lib['origin'].nunique(), 'origins')
lib.groupby(['type', 'origin']).size()

In [ ]:
## our test dataset
col2rename = {'Thermodynamic Solubility: Thermodynamic Solubility (1) (μM)':'label',
              'SMILES':'smiles',
              'Molecule Name':'compound'}
test = pd.read_csv('data/20260625_thermoSol.csv').rename(columns=col2rename)[['compound','smiles','label']]
test['label'] = test['label'].str.extract(r'(\d+\.?\d*)').astype(float)
test_MF = rdkit_tools.compute_H236_features(test, v=False)
print(test.shape)

if SOL_OVERWRITE:
    test.to_parquet('autoresearch/predict_solubility/test.parquet')
    test_MF.to_parquet('autoresearch/predict_solubility/test_MF_features.parquet')
else:
    test_MF = pd.read_parquet('autoresearch/predict_solubility/test_MF_features.parquet')
    
test.head(1)

## 1. Model performance assessment

In [ ]:
## get best public dataset:
best_types = ['thermodynamic_experimental', 'kinetic_experimental']
best_public = lib[lib['type'].isin(best_types)].reset_index(drop=True)
print(best_public.shape, '| origins:', best_public['origin'].nunique())
print(best_public['type'].value_counts())

In [ ]:
## XGB:
params = {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'min_child_weight': 1, 'scale_pos_weight': 1}
model = XGBClassifier(**params,n_jobs=-1,random_state=42)
## Champion:
champion = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, class_weight='balanced', random_state=42)


In [ ]:
## (1a) random 5-fold — INTERNAL ONLY
ML_data = pd.merge(test,test_MF).drop_duplicates('smiles')
ML_data['label'] = (ML_data['label'] >= CUTOFF_SOL) * 1
stats_tools.check_ML_data(ML_data,verbose=False)
trained_model, df_pred = ML_Class.run_K_Fold_Xval_Classification(
    ML_data, ID='compound', model=champion, folds=5,
    col_to_rm=['compound', 'label','smiles'], v=True, ctf=0.5, impute_by_mean=False)

In [ ]:
## (1b) random 5-fold — PUBLIC + INTERNAL (public fixed in train, 5 folds over in-house)
internal = pd.merge(test, test_MF).drop_duplicates('smiles')
public   = pd.merge(best_public.rename(columns={'solubility': 'label'})[['compound', 'smiles', 'label']],
                    MF_features_train).drop_duplicates('smiles')
ML_data  = pd.concat([public, internal], ignore_index=True)
ML_data['label'] = (ML_data['label'] >= CUTOFF_SOL) * 1

pub_ids, int_ids = list(public['compound']), internal['compound'].values
ID_sets = [[pub_ids + list(int_ids[tr]), list(int_ids[te])]
           for tr, te in KFold(5, shuffle=True, random_state=42).split(int_ids)]

trained_model, df_pred = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_data, ID='compound', ID_sets=ID_sets, model=champion,
    col_to_rm=['compound', 'label', 'smiles'], v=True, ctf=0.5)


In [ ]:
## (2a) temporal (oldest 70% -> newest 30%) — INTERNAL ONLY
ML_data = pd.merge(test, test_MF).drop_duplicates('smiles')
ML_data['label'] = (ML_data['label'] >= CUTOFF_SOL) * 1
order = list(ML_data.sort_values('compound', key=lambda s: s.str.extract(r'(\d+)')[0].astype(int))['compound'])
cut = int(len(order) * 0.5)
ID_sets = [[order[:cut], order[cut:]]]

trained_model, df_pred = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_data, ID='compound', ID_sets=ID_sets, model=champion,
    col_to_rm=['compound', 'label', 'smiles'], v=True, ctf=0.5)


In [ ]:
## (2b) temporal — PUBLIC + INTERNAL (public + oldest 70% internal -> newest 30% internal)
internal = pd.merge(test, test_MF).drop_duplicates('smiles')
public   = pd.merge(best_public.rename(columns={'solubility': 'label'})[['compound', 'smiles', 'label']],
                    MF_features_train).drop_duplicates('smiles')
ML_data  = pd.concat([public, internal], ignore_index=True)
ML_data['label'] = (ML_data['label'] >= CUTOFF_SOL) * 1
order = list(internal.sort_values('compound', key=lambda s: s.str.extract(r'(\d+)')[0].astype(int))['compound'])
cut = int(len(order) * 0.5)
ID_sets = [[list(public['compound']) + order[:cut], order[cut:]]]

trained_model, df_pred = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_data, ID='compound', ID_sets=ID_sets, model=champion,
    col_to_rm=['compound', 'label', 'smiles'], v=True, ctf=0.5)

In [ ]:
ppv_df = ML_Class.get_PPV_vs_proba(df_pred, npts=30, plot=True,baseline=internal['label'].sum()/internal.shape[0],dpi=80)

In [ ]:
## fit model to all dataset:
internal = pd.merge(test, test_MF).drop_duplicates('smiles')
public   = pd.merge(best_public.rename(columns={'solubility': 'label'})[['compound', 'smiles', 'label']],
                    MF_features_train).drop_duplicates('smiles')
ML_data  = pd.concat([public, internal], ignore_index=True)
ML_data['label'] = (ML_data['label'] >= CUTOFF_SOL) * 1

X, y = ML_data.drop(columns=['compound', 'label', 'smiles']), ML_data['label']
model.fit(X, y)
print('fitted champion on', X.shape, '| positive fraction %.3f' % y.mean())


## 2. Predictions

In [ ]:
## get library:
if CHEMLIB_OVERWRITE:
    serac_df = (get_df(vault=7108, collections=['AK','AJ','AA','AO','AP','AQ','AR','AS','AT','AU'], columns=['name', 'smiles']).rename(columns={'name': 'compound'}))
    serac_df.to_csv(CHEMLIB_PATH,sep=',',index=False)
else:
    serac_df = pd.read_csv(CHEMLIB_PATH)

In [ ]:
clean = pd.read_excel('data/sticky_compounds/sticky_compounds.xlsx', sheet_name='clean',header=None, names=['compound']).assign(type='clean')
problematic = pd.read_excel('data/sticky_compounds/sticky_compounds.xlsx', sheet_name='problematic',header=None, names=['compound']).assign(type='problematic')
validated_problematic = pd.read_excel('data/sticky_compounds/sticky_compounds.xlsx', sheet_name='validated_problematic',header=None, names=['compound']).assign(type='validated_problematic')
sticky = pd.concat([clean,problematic,validated_problematic]).reset_index(drop=True)
sticky_smi = pd.merge(sticky,serac_df).drop_duplicates().reset_index(drop=True)
assert(sticky.shape[0] == sticky_smi.shape[0])


In [ ]:
sticky_prop = pd.merge(sticky_smi,rdkit_tools.compute_properties_from_smiles(sticky_smi))
MF_features_sticky = rdkit_tools.compute_H236_features(sticky_smi, v=True)

In [ ]:
## Apply trained model to predict solubility
sticky_pred = MF_features_sticky[['compound']].copy()
sticky_pred['p_soluble']    = champion.predict_proba(MF_features_sticky[champion.feature_names_in_])[:, 1]
sticky_pred['pred_soluble'] = (sticky_pred['p_soluble'] >= 0.5) * 1
sticky_pred = sticky_pred.merge(sticky_prop).drop_duplicates()
# sticky_pred.groupby('type')['p_soluble'].agg(['mean', 'median', 'count'])
sticky_pred.head(3)


In [ ]:
## plotting box plots of properties vs
l = 'p_soluble'
groups = ['clean', 'problematic', 'validated_problematic']
data = [sticky_pred[sticky_pred['type'] == g][l].values for g in groups]
stats_tools.plot_nice_violinplot(data, titles=groups, show_stars=True)